# End-to-End E-Commerce Data Analytics Project
### Sales Performance & Logistics Profitability Analysis (Brazilian E-Commerce by Olist)

**Tools Used:** PostgreSQL, Python, Power BI.

---

## Business Problem Statement

Our objective is to analyze the sales performance, operational efficiency, and customer retention of Olist, a Brazilian e-commerce marketplace. As the company scales, balancing logistical costs with customer lifetime value becomes critical for sustainable profitability.

Through this End-to-End data pipeline, we aim to answer the following core business questions:

1. **Revenue Drivers:** Which product categories generate the highest gross revenue and provide a stable commercial foundation?
2. **Logistics Impact (Freight Ratio):** How do freight costs affect specific product niches? Are there categories where shipping costs disproportionately consume the product value?
3. **Payment Behavior:** Does the Brazilian installment payment system actively stimulate larger purchases and increase the Average Order Value (AOV)?
4. **Customer Retention (RFM):** What is the current state of customer loyalty? Can we segment the user base based on purchasing behavior (Recency, Frequency, Monetary) to distinguish our most valuable returning clients from one-time buyers?

## Step 0: Environment Setup & Dependencies
Before connecting to the database and processing the data, we need to ensure all required Python libraries are installed in our environment.

In [1]:
!pip install sqlalchemy psycopg2-binary

import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sqlalchemy import text

## Step 1: Database Setup & Schema Creation (PostgreSQL)
Before connecting via Python, we first established our Data Warehouse structure in PostgreSQL. The raw CSV files from Kaggle were imported into a relational database using the schema below.

This approach allows us to utilize strict data types (e.g., `TIMESTAMP` instead of flat text) and sets the foundation for our analytical views later in the project.
![Olist Database Schema](olistschema.png)
*Source: Official Olist e-commerce dataset architecture (via Kaggle).*
```sql

-- 1. Customers table (Dimension)
CREATE TABLE IF NOT EXISTS customers (
    customer_id VARCHAR(50) PRIMARY KEY,
    customer_unique_id VARCHAR(50) NOT NULL,
    customer_zip_code_prefix INT,
    customer_city VARCHAR(100),
    customer_state VARCHAR(5)
);

-- 2. Sellers table (Dimension)
CREATE TABLE IF NOT EXISTS sellers (
    seller_id VARCHAR(50) PRIMARY KEY,
    seller_zip_code_prefix INT,
    seller_city VARCHAR(100),
    seller_state VARCHAR(5)
);

-- 3. Products table (Dimension)
CREATE TABLE IF NOT EXISTS products (
    product_id VARCHAR(50) PRIMARY KEY,
    product_category_name VARCHAR(100),
    product_name_lenght FLOAT,
    product_description_lenght FLOAT,
    product_photos_qty FLOAT,
    product_weight_g FLOAT,
    product_length_cm FLOAT,
    product_height_cm FLOAT,
    product_width_cm FLOAT
);

-- 4. Product Category Translation table (Dictionary)
CREATE TABLE IF NOT EXISTS product_category_name_translation (
    product_category_name VARCHAR(100) PRIMARY KEY,
    product_category_name_english VARCHAR(100)
);

-- 5. Geolocation table (Dimension)
CREATE TABLE IF NOT EXISTS geolocation (
    geolocation_zip_code_prefix INT,
    geolocation_lat FLOAT,
    geolocation_lng FLOAT,
    geolocation_city VARCHAR(100),
    geolocation_state VARCHAR(5)
);

-- 6. Orders table (Fact)
CREATE TABLE IF NOT EXISTS orders (
    order_id VARCHAR(50) PRIMARY KEY,
    customer_id VARCHAR(50) REFERENCES customers(customer_id),
    order_status VARCHAR(20),
    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP
);

-- 7. Order Items table (Fact)
CREATE TABLE IF NOT EXISTS order_items (
    order_id VARCHAR(50) REFERENCES orders(order_id),
    order_item_id INT,
    product_id VARCHAR(50) REFERENCES products(product_id),
    seller_id VARCHAR(50) REFERENCES sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price DECIMAL(10,2),
    freight_value DECIMAL(10,2)
);

-- 8. Order Payments table (Fact)
CREATE TABLE IF NOT EXISTS order_payments (
    order_id VARCHAR(50) REFERENCES orders(order_id),
    payment_sequential INT,
    payment_type VARCHAR(20),
    payment_installments INT,
    payment_value DECIMAL(10,2)
);

-- 9. Order Reviews table (Fact)
CREATE TABLE IF NOT EXISTS order_reviews (
    review_id VARCHAR(50),
    order_id VARCHAR(50) REFERENCES orders(order_id),
    review_score INT,
    review_comment_title VARCHAR(255),
    review_comment_message TEXT,
    review_creation_date TIMESTAMP,
    review_answer_timestamp TIMESTAMP
);

## Step 2: Database Connection & Verification 

With the PostgreSQL data warehouse established, we initialize a secure connection using SQLAlchemy. In this step, we safely verify communication with the local server using a context manager, preparing the engine for the data transformations and analytical queries that will follow.

In [2]:
DB_USER = 'postgres'
DB_PASSWORD = 'olistproject' 
DB_HOST = 'localhost'
DB_PORT = '5432'
DB_NAME = 'olist_ecommerce'

engine = create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

try:
    with engine.connect() as connection:
        print("Connection to PostgreSQL established successfully. Ready for advanced analytics.")
except Exception as e:
    print(f"Database connection failed: {e}")

Connection to PostgreSQL established successfully. Ready for advanced analytics.


## Step 3: SQL-Driven Data Transformation
While Python (Pandas) is excellent for complex analytical algorithms, processing simple joins and arithmetic on large tabular datasets in local RAM can be highly inefficient. Instead, we adopt an **ELT (Extract, Load, Transform)** architecture, pushing the transformation logic down to the PostgreSQL engine. 

By creating Analytical Views directly in the database rather than transforming DataFrames in Python, we achieve:
* **Memory & Storage Efficiency:** Views are saved SQL queries, not physical tables, meaning zero data duplication.
* **Direct BI Integration:** Power BI can query these views natively, ensuring the dashboard always reflects the latest database state without needing a Python script as a middleman.

We are establishing two core views:
1. **dim_products_en:** A dimension view that uses a `LEFT JOIN` to translate categories and explicitly handles missing labels (nulls) using `COALESCE(column, 'Unknown')`. This is a critical financial safeguard ensuring BI tools do not filter out blank transactions, which would artificially lower reported revenue.
2. **fact_order_items_metrics:** A fact view that computes financial metrics (`total_item_value` and `freight_ratio`) at the database level using standard SQL arithmetic. We employ `NULLIF()` to safely handle potential division-by-zero errors.

In [3]:
create_products_view = """
CREATE OR REPLACE VIEW dim_products_en AS
SELECT 
    p.product_id,
    COALESCE(t.product_category_name_english, 'Unknown') AS product_category_name_english,
    p.product_weight_g,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm
FROM products p
LEFT JOIN product_category_name_translation t 
    ON p.product_category_name = t.product_category_name;
"""

create_order_items_view = """
CREATE OR REPLACE VIEW fact_order_items_metrics AS
SELECT 
    order_id,
    order_item_id,
    product_id,
    seller_id,
    price,
    freight_value,
    (price + freight_value) AS total_item_value,
    ROUND((freight_value / NULLIF(price, 0))::numeric, 4) AS freight_ratio
FROM order_items;
"""

with engine.begin() as connection:
    connection.execute(text(create_products_view))
    connection.execute(text(create_order_items_view))
    print("SQL Views successfully created in PostgreSQL.")


SQL Views successfully created in PostgreSQL.


## Step 4: Final SQL Aggregation
One order can have multiple payment methods (e.g., credit card + voucher / split payments), which creates multiple rows per `order_id` in the `order_payments` table. If joined directly in Power BI, this granularity mismatch would artificially inflate our total revenue. 

To prevent this Cartesian product, we create a final aggregated view in PostgreSQL (`fact_payments_agg`). We group by `order_id`, sum the total payment value, and use `MAX(payment_installments)` to capture the maximum installment count chosen for the transaction (safely handling cases where a voucher of 1 installment is mixed with a multi-installment credit card payment).

In [4]:
create_payments_view = """
CREATE OR REPLACE VIEW fact_payments_agg AS
SELECT 
    order_id,
    MAX(payment_installments) AS max_installments,
    SUM(payment_value) AS total_payment_value
FROM order_payments
GROUP BY order_id;
"""

with engine.begin() as connection:
    connection.execute(text(create_payments_view))
    print("Payment Aggregation View successfully created in PostgreSQL.")
    

Payment Aggregation View successfully created in PostgreSQL.


## Step 5: Advanced Analytics - Payment Behavior vs. AOV
Our third business objective is to determine if the Brazilian installment payment system drives higher Average Order Value (AOV). 

**Architectural Choice (ELT + Python):** 
We continue our ELT approach by using PostgreSQL to group and aggregate the average order value per installment tier. Python is then used strictly for its strength: statistical modeling.

**Handling Outliers (Why max 12 installments?):**
In the Brazilian e-commerce market, standard credit card installment plans are capped at 10 or 12 months. Data profiling reveals that transactions with >12 installments are extreme edge cases. We explicitly filter our analysis (`WHERE max_installments BETWEEN 1 AND 12`) to ensure our calculations reflect genuine consumer behavior.

**Statistical Integrity (Why use raw data for correlation?):**
We calculate the Pearson correlation coefficient using the granular, unaggregated dataset rather than the grouped averages. Calculating correlation on aggregated data averages out individual variance and artificially inflates the result (a statistical error known as the Ecological Fallacy). Using raw data reveals the true strength of the relationship between an individual customer's cart value and their chosen installment plan.

In [5]:
sql_aov = """
SELECT 
    max_installments,
    ROUND(AVG(total_payment_value)::numeric, 2) AS average_order_value,
    COUNT(order_id) AS total_orders
FROM fact_payments_agg
WHERE max_installments BETWEEN 1 AND 12
GROUP BY max_installments
ORDER BY max_installments;
"""
aov_standard = pd.read_sql_query(sql_aov, con=engine)

sql_raw_payments = """
SELECT max_installments, total_payment_value 
FROM fact_payments_agg 
WHERE max_installments BETWEEN 1 AND 12;
"""
payments_raw_df = pd.read_sql_query(sql_raw_payments, con=engine)

correlation = payments_raw_df['max_installments'].corr(payments_raw_df['total_payment_value'])

print("Average Order Value (AOV) by Installment Tier (1-12) calculated via SQL:")
print(aov_standard.to_string(index=False))
print(f"\nPearson Correlation (calculated via Python): {correlation:.3f}")

Average Order Value (AOV) by Installment Tier (1-12) calculated via SQL:
 max_installments  average_order_value  total_orders
                1               121.04         48268
                2               129.12         12363
                3               144.36         10429
                4               165.06          7070
                5               184.89          5227
                6               211.59          3908
                7               189.34          1622
                8               309.77          4251
                9               204.58           644
               10               418.75          5315
               11               124.93            23
               12               323.05           133

Pearson Correlation (calculated via Python): 0.317


## Step 6: Advanced Analytics - Customer Segmentation (RFM)
As a final business objective, we segment our customers using the RFM (Recency, Frequency, Monetary) model. We assign scores from 1 to 4 in each category, where 4 is the most optimal behavior.

**Handling Olist Dataset Quirks:**
1. **The ID Trap:** In the `orders` table, `customer_id` is just a one-time session token. To accurately track returning customers, we join the `customers` table to access the `customer_unique_id`.
2. **The Frequency Problem:** Data profiling proves that ~97% of Olist customers made only a single purchase. Because the data is so heavily skewed, standard statistical splitting (like `pd.qcut` used for Recency and Monetary) fails due to duplicate bin edges. We fix this by applying a vectorized `.clip()` logic to score Frequency.

**Tailored Business Segmentation Logic:**
Because returning customers are so rare (top ~3% of the database), our segmentation strategy is specifically adapted to Olist's business reality:
* **Champions:** Bought recently (R ≥ 3), return buyers (F ≥ 2), and spent significantly (M ≥ 3).
* **Loyal Customers:** Any returning customer (F ≥ 2) who doesn't meet the strict Champion criteria.
* **Promising New:** Bought recently (R ≥ 3) but only once (F = 1).
* **At Risk (High Value):** Bought a long time ago (R ≤ 2) and only once (F = 1), but spent a lot (M ≥ 3). This is a prime target group for win-back campaigns with discount codes.
* **Hibernating:** Bought a long time ago (R ≤ 2), only once (F = 1), and spent little (M ≤ 2).

In [16]:
rfm_query = """
SELECT 
    c.customer_unique_id,
    MAX(o.order_purchase_timestamp) AS last_purchase_date,
    COUNT(DISTINCT o.order_id) AS frequency,
    SUM(p.total_payment_value) AS monetary
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN fact_payments_agg p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_unique_id;
"""
rfm_df = pd.read_sql_query(rfm_query, con=engine)


print("Customer Order Frequency:")
freq_dist = rfm_df['frequency'].value_counts(normalize=True).head(4) * 100
print(freq_dist.round(2).astype(str) + '%')


snapshot_date = pd.to_datetime(rfm_df['last_purchase_date']).max() + pd.Timedelta(days=1)
rfm_df['recency'] = (snapshot_date - pd.to_datetime(rfm_df['last_purchase_date'])).dt.days

rfm_df['R_Score'] = pd.qcut(rfm_df['recency'], q=4, labels=[4, 3, 2, 1]).astype(int)
rfm_df['M_Score'] = pd.qcut(rfm_df['monetary'], q=4, labels=[1, 2, 3, 4]).astype(int)
rfm_df['F_Score'] = rfm_df['frequency'].clip(upper=4).astype(int)

conditions = [
    (rfm_df['R_Score'] >= 3) & (rfm_df['F_Score'] >= 2) & (rfm_df['M_Score'] >= 3),
    (rfm_df['F_Score'] >= 2) & ((rfm_df['R_Score'] < 3) | (rfm_df['M_Score'] < 3)),
    (rfm_df['R_Score'] >= 3) & (rfm_df['F_Score'] == 1),
    (rfm_df['R_Score'] <= 2) & (rfm_df['F_Score'] == 1) & (rfm_df['M_Score'] >= 3),
    (rfm_df['R_Score'] <= 2) & (rfm_df['F_Score'] == 1) & (rfm_df['M_Score'] <= 2)
]

choices = [
    'Champions',
    'Loyal Customers',
    'Promising New',
    'At Risk (High Value)',
    'Hibernating'
]

rfm_df['Business_Label'] = np.select(conditions, choices, default='Regulars')


print("\nCustomer Breakdown by Business Segment:")
print(rfm_df['Business_Label'].value_counts())


print("\nSample of 'Champions' Segment:")
champions = rfm_df[rfm_df['Business_Label'] == 'Champions']
print(champions[['customer_unique_id', 'recency', 'frequency', 'monetary', 'Business_Label']].head().to_string(index=False))

Customer Order Frequency:
frequency
1    97.0%
2    2.76%
3    0.19%
4    0.03%
Name: proportion, dtype: object

Customer Breakdown by Business Segment:
Business_Label
Promising New           45298
Hibernating             23590
At Risk (High Value)    21668
Loyal Customers          1422
Champions                1379
Name: count, dtype: int64

Sample of 'Champions' Segment:
              customer_unique_id  recency  frequency  monetary Business_Label
00a39521eb40f7012db50455bf083460       88          2    123.25      Champions
011575986092c30523ecb71ff10cb473      133          2    214.90      Champions
011b4adcd54683b480c4d841250a987f      196          2    236.30      Champions
012452d40dafae4df401bced74cdb490      108          2    495.33      Champions
012a218df8995d3ec3bb221828360c86       73          2   1510.38      Champions


## Step 7: Data Centralization for BI Integration
To maintain a seamless and efficient ELT architecture, we push the Python-calculated RFM segments back into the PostgreSQL database rather than exporting flat CSV files. Writing the results directly to a dedicated table establishes a Single Source of Truth (SSOT). This ensures Power BI can connect to one centralized location, natively querying and joining our SQL-aggregated financial views with the Python-generated statistical segments without relying on fragile external file structures.

In [ ]:
rfm_table_name = 'analytics_rfm_segments'

rfm_df.to_sql(rfm_table_name, con=engine, if_exists='replace', index=False)

print(f"RFM DataFrame successfully loaded into PostgreSQL as '{rfm_table_name}'.")


RFM DataFrame successfully loaded into PostgreSQL as 'analytics_rfm_segments'.


## Step 8: Executive Summary & BI Dashboard

This project establishes an end-to-end ELT data pipeline to analyze sales performance, logistics efficiency, and customer retention for the Brazilian Olist marketplace. By leveraging PostgreSQL for structural data aggregation, Python for advanced statistical modeling, and Power BI for interactive visualization, the architecture maintains a strict Single Source of Truth (SSOT) and delivers actionable business intelligence.

**Key Business Conclusions**
* **Revenue Leaders:** The **Health Beauty** (>1.4M BRL) and **Watches Gifts** (~1.3M BRL) categories are the primary financial drivers. Five distinct product categories successfully exceed the 1M BRL gross revenue threshold, providing a stable commercial foundation.
* **Logistics Bottlenecks:** Freight costs disproportionately impact margins in specific physical niches. Categories such as **Home Comfort** and **Dvds Blu Ray** exhibit a staggering **80-90% freight ratio**, requiring immediate renegotiation of carrier weight limits.
* **Installments vs. AOV:** Statistical modeling proves a positive correlation between the installment payment system and shopping cart value. Splitting costs over extended periods (up to 12 months) directly stimulates customers to purchase premium items, significantly increasing the Average Order Value.
* **Retention Crisis (RFM):** Custom RFM segmentation reveals a critical loyalty deficit. Exactly **48.5% of the user base** falls into the *Promising New* segment (one-time buyers), while elite returning segments form a minimal fraction. This highlights an urgent need to shift marketing budgets from acquisition to targeted "win-back" campaigns.

![Power BI Dashboard](raportolistostateczny.png)